# Dependencias

In [ ]:
import ctypes
ctypes.CDLL("/home/gcamilo/envs/wrf312/lib/libgomp.so.1", mode=ctypes.RTLD_GLOBAL)
import sys
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import time
import glob
import random
import pandas as pd
import copy
import re
import wrf
import cartopy.feature as cfeature
import cartopy.crs as crs
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
from tqdm.auto import tqdm
from datetime import datetime
from IPython.display import display
from matplotlib import animation
from matplotlib.colors import LinearSegmentedColormap
from skimage.metrics import structural_similarity as ssim
from wrf import getvar, get_cartopy, latlon_coords, to_np
from netCDF4 import Dataset as ncDataset
from scipy.ndimage import gaussian_filter1d

ruta_padre = os.path.abspath('..')
if ruta_padre not in sys.path:
    sys.path.append(ruta_padre)

CROP     = 1
# Rutas a las dos carpetas de salidas WRF (separadas por espacio en disco)
dir_2018 = os.path.expanduser("~/jupyter/salidasWRF2018")
dir_2019 = "../salidasWRF2019"
archivos = sorted(
    glob.glob(os.path.join(dir_2018, "wrfout_d01_*")) +
    glob.glob(os.path.join(dir_2019, "wrfout_d01_*"))
)
with ncDataset(archivos[0]) as ds0:
    ref = getvar(ds0, "LIGHTDENS")
    CART_PROJ = get_cartopy(ref)
    LATS, LONS = latlon_coords(ref)
LATS, LONS = to_np(LATS)[CROP:-CROP, CROP:-CROP], to_np(LONS)[CROP:-CROP, CROP:-CROP]

from modulos.tormentaDataset import TormentaDataset

# Chequeamos si hay GPU disponible para acelerar el entrenamiento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

In [ ]:
print("=== DIAGNÓSTICO DE PYTORCH Y CUDA ===")
print("Versión de PyTorch:", torch.__version__)
print("Versión de CUDA instalada en PyTorch:", torch.version.cuda)
print("¿CUDA disponible?:", torch.cuda.is_available())
print("Cantidad de GPUs detectadas:", torch.cuda.device_count())
print("=====================================\n")

if torch.cuda.is_available():
    try:
        device = torch.device("cuda")
        # Forzamos una operación matemática en la placa para validar el puente físico
        torch.zeros(1).to(device)
        print("✅ ¡Conexión exitosa a la GPU!")
        print(f"Propiedades del hardware asignado: {torch.cuda.get_device_name(0)}")
    except RuntimeError as e:
        print("❌ Error al interactuar con el hardware:", e)
else:
    print("⚠️ CUDA no está disponible en absoluto para este binario de PyTorch.")

In [ ]:
# Para REPRODUCIBILIDAD

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Ext Clip U-Net Comb V2.1

Evaluación del mejor ensayo de la Fase 2 (Etapa B) de la optimización en Clementina
(`clementina/objetivo_comb.py`, estudio `etapa_B_Ext_Clip_Comb_V21`), con los pesos de su mejor época
(la de menor pérdida de validación híbrida), igual que el resto de los modelos de la Comb.

El modelo **no se entrena acá**: el checkpoint se exporta en Clementina con
`clementina/exportar_checkpoint_comb.py` y se copia a `../modelos/`.

In [ ]:
class UNet_WRF(nn.Module):
    """
    U-Net con profundidad y ancho configurables.

    Parámetros
    ----------
    in_channels : int, default 6
        Cantidad de canales de entrada (variables WRF).
    n_niveles : int, default 2
        Cantidad de niveles de encoder/decoder (pooling + upsampling).
        La arquitectura original tenía 2 niveles.
    n_filtros_base : int, default 32
        Cantidad de filtros en el primer nivel del encoder. Cada nivel
        siguiente duplica esta cantidad (32→64→128→...), igual que el
        bottleneck, que duplica una vez más el último nivel.
    kernel_sizes : list[int] o None
        Un kernel_size por nivel del encoder (y su espejo en el decoder).
        Si es None, usa kernel_size=3 en todos los niveles.
        Debe tener longitud == n_niveles.
    dropout_ps : list[float] o None
        Un dropout_p por nivel del encoder (y su espejo en el decoder),
        aplicado también en el bottleneck (usa el último valor de la lista).
        Si es None, no aplica dropout en ningún nivel (comportamiento original).
        Debe tener longitud == n_niveles.
    input_size : int, default 297
        Tamaño espacial (H=W) de la imagen de entrada, usado para calcular
        el padding necesario según la profundidad elegida.
    """
    def __init__(self, in_channels=6, n_niveles=2, n_filtros_base=32,
                 kernel_sizes=None, dropout_ps=None, input_size=297):
        super(UNet_WRF, self).__init__()

        if kernel_sizes is None:
            kernel_sizes = [3] * n_niveles
        assert len(kernel_sizes) == n_niveles, \
            f"kernel_sizes debe tener {n_niveles} elementos, tiene {len(kernel_sizes)}"

        if dropout_ps is None:
            dropout_ps = [0.0] * n_niveles
        assert len(dropout_ps) == n_niveles, \
            f"dropout_ps debe tener {n_niveles} elementos, tiene {len(dropout_ps)}"
        
        self.n_niveles = n_niveles
        self.dropout_ps = dropout_ps

        # --- Padding espacial dinámico según profundidad ---
        divisor = 2 ** n_niveles
        size_ajustado = ((input_size // divisor) + 1) * divisor
        pad_total = size_ajustado - input_size
        # Repartimos el padding: menos a la izquierda/arriba, más a la derecha/abajo
        self.pad_izq_arr = pad_total // 2
        self.pad_der_aba = pad_total - self.pad_izq_arr
        self.input_size = input_size

        # --- Canales por nivel
        canales = [n_filtros_base * (2 ** i) for i in range(n_niveles)]
        canal_bottleneck = n_filtros_base * (2 ** n_niveles)
        # Kernel del bottleneck: reusamos el del nivel más profundo
        k_bottleneck = kernel_sizes[-1]
        dropout_bottleneck = dropout_ps[-1]

        def bloque_conv(canal_in, canal_out, k, dropout_p):
            pad = k // 2
            capas = [
                nn.Conv2d(canal_in, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
                nn.Conv2d(canal_out, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
            ]
            if dropout_p > 0:
                capas.append(nn.Dropout2d(p=dropout_p))
            return nn.Sequential(*capas)
        
        # --- ENCODER (dinámico) ---
        self.enc_blocks = nn.ModuleList()
        self.pools      = nn.ModuleList()
        canal_in = in_channels
        for canal_out, k, p in zip(canales, kernel_sizes, dropout_ps):
            self.enc_blocks.append(bloque_conv(canal_in, canal_out, k, p))
            self.pools.append(nn.MaxPool2d(2, 2))
            canal_in = canal_out

        # --- BOTTLENECK ---
        self.bottleneck = bloque_conv(canal_in, canal_bottleneck, k_bottleneck, dropout_bottleneck)

        # --- DECODER (dinámico, en orden inverso al encoder) ---
        self.ups        = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        canal_in = canal_bottleneck
        for canal_skip, k, p in zip(reversed(canales), reversed(kernel_sizes), reversed(dropout_ps)):
            self.ups.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
            self.dec_blocks.append(bloque_conv(canal_in + canal_skip, canal_skip, k, p))
            canal_in = canal_skip

        # --- SALIDA ---
        self.salida  = nn.Conv2d(canal_in, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()  # SOLO para FACL

    def forward(self, x):
        x_pad = F.pad(x, (self.pad_izq_arr, self.pad_der_aba,
                           self.pad_izq_arr, self.pad_der_aba))

        # --- Encoder ---
        skips = []
        h = x_pad
        for bloque, pool in zip(self.enc_blocks, self.pools):
            h = bloque(h)
            skips.append(h)       # guardamos para el skip connection
            h = pool(h)

        # --- Bottleneck ---
        h = self.bottleneck(h)

        # --- Decoder ---
        for up, bloque, skip in zip(self.ups, self.dec_blocks, reversed(skips)):
            h = up(h)
            h = torch.cat([h, skip], dim=1)
            h = bloque(h)

        out = self.salida(h)
        out = self.sigmoid(out)  # SOLO para FACL

        # Recortar el padding agregado al principio
        out = out[:, :, self.pad_izq_arr:self.pad_izq_arr + self.input_size,
                         self.pad_izq_arr:self.pad_izq_arr + self.input_size]
        return out

In [ ]:
# ==========================================
# FUNCIÓN DE PÉRDIDA
# ==========================================
# CombV21: copia de clementina/perdida_comb.py (la misma clase con la que se entrenó y validó en Clementina).
#   val = (WMSE + sqrt(HW)*FAL + sqrt(HW)*FCL) / 3, acumulada sobre todo el set.
class CombV21(nn.Module):
    def __init__(self, desnormalizar_y, peso_positivo, max_prob, epocas_para_max_prob, pasos_por_epoca,
                 min_prob=0.0, eps=1e-8, rampa_exacta=False):
        super().__init__()
        assert 0.0 <= min_prob < max_prob <= 1.0, (min_prob, max_prob)
        self.desnormalizar_y = desnormalizar_y
        self.peso_positivo = float(peso_positivo)
        self.min_prob, self.max_prob = float(min_prob), float(max_prob)
        self.prob_slope = (self.max_prob - self.min_prob) / (max(epocas_para_max_prob, 1e-9) * pasos_por_epoca)
        self.rampa_exacta = rampa_exacta
        self.eps = eps
        self.step = 0
        self.ultimo_termino = None
        self.val_reset()

    def prob_wmse(self, t=None):
        t = self.step if t is None else t
        if self.rampa_exacta:
            p = self.min_prob + self.prob_slope * t
        else:
            p = (t * self.prob_slope) * (self.max_prob - self.min_prob) + self.min_prob
        return min(p, self.max_prob)

    @staticmethod
    def _fft(x):
        return torch.fft.fftn(x.float().squeeze(1), dim=[-1, -2], norm='ortho')

    def fal(self, fp, fg):
        return ((fp.abs() - fg.abs()) ** 2).mean()

    def fcl(self, fp, fg):
        dims = tuple(range(1, fp.dim()))
        num = torch.sum((fp * torch.conj(fg)).real, dim=dims)
        n_p = torch.sqrt(torch.sum(fp.abs() ** 2, dim=dims))
        n_g = torch.sqrt(torch.sum(fg.abs() ** 2, dim=dims))
        return (1 - num / (n_p * n_g + self.eps)).mean()

    def wmse(self, pred, gt):
        p, g = self.desnormalizar_y(pred.float()), self.desnormalizar_y(gt.float())
        w = torch.where(g > 0, torch.full_like(g, self.peso_positivo), torch.ones_like(g))
        return ((p - g) ** 2 * w).mean()

    @staticmethod
    def _peso_fourier(pred):
        H, W = pred.shape[-2:]
        return float(np.sqrt(H * W))

    # ---- validación sobre todo el set ----
    def val_reset(self):
        self._v = {'wmse': 0.0, 'fal': 0.0, 'fcl': 0.0, 'n': 0, 'w': 1.0}

    @torch.no_grad()
    def val_acumular(self, pred, gt):
        fp, fg = self._fft(pred), self._fft(gt)
        n = pred.shape[0]
        self._v['wmse'] += float(self.wmse(pred, gt)) * n
        self._v['fal'] += float(self.fal(fp, fg)) * n
        self._v['fcl'] += float(self.fcl(fp, fg)) * n
        self._v['n'] += n
        self._v['w'] = self._peso_fourier(pred)

    def val_resultado(self):
        v = self._v
        self.val_wmse, self.val_fal, self.val_fcl = (v[k] / v['n'] for k in ('wmse', 'fal', 'fcl'))   # FAL y FCL sin escalar
        return float((self.val_wmse + v['w'] * self.val_fal + v['w'] * self.val_fcl) / 3.0)

    def forward(self, pred, gt, mode='train'):
        if mode == 'val':      # un solo lote (para el set completo: val_reset / val_acumular / val_resultado)
            fp, fg = self._fft(pred), self._fft(gt)
            w = self._peso_fourier(pred)
            return (self.wmse(pred, gt) + w * self.fal(fp, fg) + w * self.fcl(fp, fg)) / 3.0
        if np.random.rand() < self.prob_wmse():
            loss, self.ultimo_termino = self.wmse(pred, gt), 'WMSE'
        else:
            fp, fg = self._fft(pred), self._fft(gt)
            w = self._peso_fourier(pred)
            if np.random.rand() < 0.5:
                loss, self.ultimo_termino = self.fal(fp, fg) * w, 'FAL'
            else:
                loss, self.ultimo_termino = self.fcl(fp, fg) * w, 'FCL'
        self.step += 1
        return loss


In [ ]:
# =====================================================================
# DATASETS
# =====================================================================

ruta_train = '../dataset/train'
ruta_val   = '../dataset/val'
ruta_stats = '../dataset/stats.npz'

print("Mapeando archivos y cargando estadísticas globales...")
train_dataset = TormentaDataset(split_dir=ruta_train, stats_path=ruta_stats)
val_dataset   = TormentaDataset(split_dir=ruta_val,   stats_path=ruta_stats)
print(f"-> Train: {len(train_dataset)} muestras | Val: {len(val_dataset)} muestras")
print(f"-> min_y: {train_dataset.min_y:.4f} | max_y: {train_dataset.max_y:.4f}")

vacias = 0
for _, y in DataLoader(train_dataset, batch_size=64, shuffle=False):
    vacias += (y.flatten(1).amax(dim=1) < 1e-6).sum().item()
print(f"-> Targets vacíos en train: {vacias} / {len(train_dataset)} ({100*vacias/len(train_dataset):.2f}%)")

# El modelo se entrenó con el conjunto extendido y RESTRINGIDO (LIGHTDENS <= 150) y 3 canales
assert train_dataset[0][0].shape[0] == 3, "el dataset tiene que ser el de 3 canales (MDBZ, HIELO, CTT)"
y_max = max(val_dataset.desnormalizar_y(val_dataset[i][1]).max().item() for i in range(0, len(val_dataset), 25))
print(f"-> LIGHTDENS máximo (muestra de val): {y_max:.2f}  (tiene que ser <= 150: conjunto restringido)")
assert y_max <= 150.01, "../dataset no es el conjunto restringido (Clip)"


In [ ]:
# =====================================================================
# HIPERPARÁMETROS (Fase 2, ensayo 92: los exporta exportar_checkpoint_comb.py en el checkpoint)
# =====================================================================
NOMBRE_MODELO   = 'Ext Clip U-Net Comb V2.1'
NOMBRES_CANALES = ['MDBZ', 'HIELO', 'CTT']
TRIAL           = 92
RUTA_CKPT       = f'../modelos/B_trial{TRIAL:04d}_{NOMBRE_MODELO}.pth'

checkpoint = torch.load(RUTA_CKPT, map_location=device, weights_only=False)   # checkpoint propio, de confianza
BATCH_SIZE = checkpoint['batch_size']            # 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16,         shuffle=False, num_workers=0, pin_memory=True)  # BATCH_VAL de Clementina

# Arquitectura de la Fase 1 (estudios/arq_comb_A.json): mejor ensayo (98), dropout redondeado a 2 decimales
modelo = UNet_WRF(
    in_channels=3,
    n_niveles=3,
    n_filtros_base=32,
    kernel_sizes=[3, 5, 3],
    dropout_ps=[0.01, 0.16, 0.24]
).to(device)
n_par = sum(p.numel() for p in modelo.parameters())
assert n_par == 2_307_329, n_par

criterion = CombV21(val_dataset.desnormalizar_y, peso_positivo=checkpoint['w_pos'], max_prob=checkpoint['max_prob'],
                    epocas_para_max_prob=checkpoint['rampa_epocas'], pasos_por_epoca=len(train_loader),
                    rampa_exacta=True).to(device)

print(f"-> Modelo:      {NOMBRE_MODELO} (ensayo {checkpoint['trial']}, {n_par:,} parámetros)")
print(f"-> lr {checkpoint['lr']:.3e} | lote {BATCH_SIZE} | MAX_PROB {checkpoint['max_prob']:.3f} | "
      f"max_norm {checkpoint['max_norm']:.1f} | w+ {checkpoint['w_pos']} | rampa {checkpoint['rampa_epocas']} épocas")
print(f"-> Mejor época: {checkpoint['mejor_epoca']} de {checkpoint['epochs_entrenadas']} "
      f"(val {checkpoint['mejor_val_loss']:.5f}) | early stopping con paciencia {checkpoint['patience']}")


In [ ]:
print("\n\n=== RESUMEN DE LA U-NET ===")
summary(modelo, input_size=(1, 3, 297, 297),
        col_names=["input_size", "output_size", "num_params"])

In [ ]:
# Pesos de la MEJOR época (criterio de la metodología: el best del mejor ensayo)
modelo.load_state_dict(checkpoint['best_model_state_dict'])
modelo.eval()

# Chequeo: la validación recalculada acá tiene que coincidir con la de Clementina
criterion.val_reset()
with torch.no_grad():
    for X, y in tqdm(val_loader, desc='Validación'):
        criterion.val_acumular(modelo(X.to(device)), y.to(device))
val = criterion.val_resultado()
print(f"val recalculada {val:.5f} | Clementina {checkpoint['mejor_val_loss']:.5f} "
      f"(WMSE {criterion.val_wmse:.2f} | FAL {criterion.val_fal:.3e} | FCL {criterion.val_fcl:.4f})")
assert abs(val - checkpoint['mejor_val_loss']) / checkpoint['mejor_val_loss'] < 1e-3, \
    "no coincide: revisar dataset, pesos o pérdida"


# Resultados

In [ ]:
def crear_cmap_lightdens(n_colores=256, ancho_transicion=30):
    """
    Colormap basado en 'hot_r', pero con el valor mínimo forzado a blanco
    puro. Los primeros `ancho_transicion` colores interpolan linealmente
    desde blanco hasta el color que 'hot_r' ya tiene en ese punto de la
    rampa, así la transición es continua (sin salto ni tramo plano).
    A partir de ahí, el colormap sigue siendo 'hot_r' sin modificar.
    """
    base = plt.get_cmap('hot_r', n_colores)
    colores = base(np.linspace(0, 1, n_colores))

    for i in range(ancho_transicion):
        t = i / (ancho_transicion - 1)
        colores[i] = (1 - t) * np.array([1, 1, 1, 1]) + t * colores[i]

    return LinearSegmentedColormap.from_list('lightdens_white', colores, N=n_colores)

CMAP_LIGHTDENS = crear_cmap_lightdens()

In [ ]:
def resumen_entrenamiento(checkpoint, nombre_modelo, train_loader, val_loader, batch_size):
    """
    Imprime un resumen del entrenamiento combinando datos del checkpoint
    (guardados durante el entrenamiento) con datos del DataLoader actual
    (que no se guardan en el .pth).
    """
    tiempo_total_seg = sum(checkpoint['historial_tiempo_epoch'])
    horas = int(tiempo_total_seg // 3600)
    minutos = int((tiempo_total_seg % 3600) // 60)
    tiempo_str = f"{horas}h {minutos}m" if horas > 0 else f"{minutos}m"

    print(f"-> Modelo Activo:  {nombre_modelo}")
    print(f"-> Épocas totales: {checkpoint['epochs_entrenadas']}")
    print(f"-> Learning Rate:  {checkpoint['lr']}")
    print(f"-> Batch Size:     {batch_size}")
    print(f"-> Archivos para entrenar (Train): {len(train_loader.dataset)}")
    print(f"-> Archivos para validar (Val): {len(val_loader.dataset)}")
    print(f"-> Batches por época (Train): {len(train_loader)}")
    print(f"-> Tiempo: {tiempo_str}")
    print(f"-> Mejor val_loss: {checkpoint['mejor_val_loss']:.6f} (época {checkpoint['mejor_epoca']})")
    print(f"-> Patience: {checkpoint['patience']}")

def generar_ruta_imagen(carpeta, prefijo, idx=None):
    """
    Genera la ruta de guardado. Si se provee un 'idx', lo incluye en el nombre.
    Si el archivo ya existe, le anexa un contador (_01, _02, etc.) para no pisarlo.
    """
    os.makedirs(carpeta, exist_ok=True)
    
    # Armamos el nombre base. Ej: "reconstruccion_unet_idx_42" o "curva_error_unet"
    if idx is not None:
        nombre_base = f"{prefijo}_idx_{idx}"
    else:
        nombre_base = prefijo
        
    ruta = os.path.join(carpeta, f"{nombre_base}.png")
    
    # Si no existe, usamos ese nombre directamente
    if not os.path.exists(ruta):
        return ruta
        
    # Si ya existe, empezamos a probar con un contador hasta encontrar uno libre
    contador = 1
    while True:
        ruta_con_contador = os.path.join(carpeta, f"{nombre_base}_{contador:02d}.png")
        if not os.path.exists(ruta_con_contador):
            return ruta_con_contador
        contador += 1

def graficar_curva_error(checkpoint, nombre_modelo='Modelo'):
    """
    Grafica la curva de MSE para train y validación a partir de un checkpoint cargado.
    """
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    mejor_epoca    = checkpoint['mejor_epoca']
    mejor_val_loss = checkpoint['mejor_val_loss']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epocas, historial_train, 'o-', label='Train Loss (HFACLV2.1)', color='steelblue')
    ax.plot(epocas, historial_val,   'o-', label='Val Loss (HFACLV2.1)',   color='#722F37')
    
    ax.scatter([mejor_epoca], [mejor_val_loss], s=180, marker='*',
               color='gold', edgecolor='black', zorder=5,
               label=f'Mejor época ({mejor_epoca})')
    ax.axvline(mejor_epoca, color='gray', linestyle='--', alpha=0.5)
    ax.annotate(f'{mejor_val_loss:.4f}',
                xy=(mejor_epoca, mejor_val_loss),
                xytext=(8, 10), textcoords='offset points',
                fontsize=9, fontweight='bold')
    
    ax.set_xlabel('Época')
    ax.set_ylabel('HFACLV2.1 Loss')
    ax.set_title(f'Curva de aprendizaje — {nombre_modelo}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"curva_error_{nombre_limpio}")
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica de error guardada en: {ruta_guardado}")
    
    plt.show()
    
    mejor_epoca = historial_val.index(min(historial_val)) + 1
    print(f"Mejor val_loss: {min(historial_val):.6f} (época {mejor_epoca})")

def graficar_reconstruccion(modelo, dataset, nombre_modelo='Modelo', canal_entrada=0, idx=None):
    VAR_NOMBRES = NOMBRES_CANALES

    if idx is None:
        idx = random.randint(0, len(dataset) - 1)
    print(f"Timestep seleccionado: {idx} ({dataset.files[idx].name})")

    modelo.eval()
    X, y = dataset[idx]
    with torch.no_grad():
        pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

    y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
    y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())
    entrada = X[canal_entrada].numpy()

    # El real nunca puede superar 150 (clippeado en el dataset) — cualquier
    # exceso es ruido de float. La predicción sí puede superar 150 legítimamente.
    real_max = min(y_real.max(), 150.0)
    pred_max = round(float(y_pred.max()), 1)

    vmax_crudo = max(real_max, pred_max)
    if vmax_crudo <= 0:
        vmax_crudo = 1.0

    # Step "lindo" para los ticks de la colorbar
    paso_base = max(1, int(np.ceil(vmax_crudo / 7.0)))
    if paso_base >= 10:
        x = int(np.ceil(paso_base / 10.0)) * 10
    elif paso_base > 5:
        x = int(np.ceil(paso_base / 5.0)) * 5
    else:
        x = paso_base
    vmax_real = int(np.ceil(vmax_crudo / x)) * x
    ticks = np.arange(0, vmax_real + x, x)

    # Extensión del dominio y fondo (mismo criterio que la animación)
    ext_lon_min, ext_lon_max = float(np.min(LONS)), float(np.max(LONS))
    ext_lat_min, ext_lat_max = float(np.min(LATS)), float(np.max(LATS))

    def dibujar_fondo(ax):
        ax.set_extent([ext_lon_min, ext_lon_max, ext_lat_min, ext_lat_max], crs=crs.PlateCarree())
        ax.add_feature(cfeature.BORDERS, linewidth=0.8)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 8, 'color': 'gray'}
        gl.ylabel_style = {'size': 8, 'color': 'gray'}

    fig = plt.figure(figsize=(16, 5), facecolor='white')
    fig.suptitle(f'{nombre_modelo} — Timestep {idx}', fontsize=13)

    ax0 = fig.add_subplot(1, 3, 1, projection=CART_PROJ)
    dibujar_fondo(ax0)
    cf0 = ax0.contourf(LONS, LATS, entrada, transform=crs.PlateCarree(), cmap='Blues')
    ax0.set_title(f'Entrada: {VAR_NOMBRES[canal_entrada]}')
    fig.colorbar(cf0, ax=ax0, fraction=0.046, pad=0.04)

    ax1 = fig.add_subplot(1, 3, 2, projection=CART_PROJ)
    dibujar_fondo(ax1)
    cf1 = ax1.pcolormesh(LONS, LATS, y_real, transform=crs.PlateCarree(),
                          cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax_real, shading='auto')
    ax1.set_title('Real: LIGHTDENS')
    fig.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04, ticks=ticks)

    ax2 = fig.add_subplot(1, 3, 3, projection=CART_PROJ)
    dibujar_fondo(ax2)
    cf2 = ax2.pcolormesh(LONS, LATS, y_pred, transform=crs.PlateCarree(),
                          cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax_real, shading='auto')
    ax2.set_title(f'Predicción: {nombre_modelo}')
    fig.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04, ticks=ticks)

    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"reconstruccion_{nombre_limpio}", idx=idx)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Reconstrucción guardada en: {ruta_guardado}")

    plt.show()

    print(f"Real  — min: {y_real.min():.2f}, max: {y_real.max():.2f}, mean: {y_real.mean():.4f}")
    print(f"Pred  — min: {y_pred.min():.2f}, max: {y_pred.max():.2f}, mean: {y_pred.mean():.4f}")

def agrupar_indices_por_tormenta(dataset):
    """
    Agrupa los índices de `dataset` (TormentaDataset) en tormentas: tramos de
    archivos consecutivos en el tiempo (mismo paso que el resto del split).
    Como el dataset de val solo tiene los samples de las tormentas de
    validación (sin los timesteps de las tormentas de train intercaladas),
    un salto de tiempo entre dos archivos consecutivos marca el límite entre
    una tormenta y la siguiente. No depende de una lista de fechas mantenida
    a mano: si val_tormentas cambia, esto se ajusta solo.

    Devuelve dict id_tormenta -> lista de índices (ordenados), donde
    id_tormenta es la fecha/hora del primer archivo ('YYYY-MM-DD_HHMM').
    """
    n = len(dataset)
    if n == 0:
        return {}
    ts = []
    for idx in range(n):
        m = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{2})-(\d{2})-(\d{2})', dataset.files[idx].name)
        fecha, h, mi, s = m.group(1), m.group(2), m.group(3), m.group(4)
        ts.append((idx, datetime.strptime(f"{fecha} {h}:{mi}:{s}", "%Y-%m-%d %H:%M:%S")))
    ts.sort(key=lambda x: x[1])
    if n == 1:
        return {f"{ts[0][1]:%Y-%m-%d_%H%M}": [ts[0][0]]}

    paso = min(b[1] - a[1] for a, b in zip(ts[:-1], ts[1:]))
    grupos = {}
    grupo_actual = [ts[0]]
    for a, b in zip(ts[:-1], ts[1:]):
        if b[1] - a[1] == paso:
            grupo_actual.append(b)
        else:
            clave = f"{grupo_actual[0][1]:%Y-%m-%d_%H%M}"
            grupos[clave] = [i for i, _ in grupo_actual]
            grupo_actual = [b]
    clave = f"{grupo_actual[0][1]:%Y-%m-%d_%H%M}"
    grupos[clave] = [i for i, _ in grupo_actual]
    return grupos

def animar_reconstruccion_por_tormenta(modelo, dataset, nombre_modelo='Model', fps=5, umbral_ruido=0.5):
    modelo.eval()

    grupos_por_tormenta = agrupar_indices_por_tormenta(dataset)
    rutas_generadas = []

    for tormenta_id, indices_tormenta in grupos_por_tormenta.items():
        print(f"\n=== Generating animation for {tormenta_id} ({len(indices_tormenta)} timesteps) ===")

        n_frames = len(indices_tormenta)
        predicciones_cache = []
        reales_cache = []

        for idx in tqdm(indices_tormenta, desc=f"Calculating range — {tormenta_id}"):
            X, y = dataset[idx]
            with torch.no_grad():
                pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

            y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
            y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

            y_pred[y_pred < umbral_ruido] = np.nan
            y_real[y_real < umbral_ruido] = np.nan

            reales_cache.append(y_real)
            predicciones_cache.append(y_pred)

        reales_validos = np.concatenate([r[~np.isnan(r)] for r in reales_cache])
        preds_validos  = np.concatenate([p[~np.isnan(p)] for p in predicciones_cache])

        real_max = min(reales_validos.max(), 150.0) if reales_validos.size > 0 else 0.0
        pred_max = round(float(preds_validos.max()), 1) if preds_validos.size > 0 else 0.0

        vmax_crudo = max(real_max, pred_max)
        if vmax_crudo <= 0:
            vmax_crudo = 10.0

        paso_base = max(1, int(np.ceil(vmax_crudo / 7.0)))
        if paso_base >= 10:
            x = int(np.ceil(paso_base / 10.0)) * 10
        elif paso_base > 5:
            x = int(np.ceil(paso_base / 5.0)) * 5
        else:
            x = paso_base
        vmax = int(np.ceil(vmax_crudo / x)) * x
        ticks = np.arange(0, vmax + x, x)

        print(f"LIGHTDENS Range ({tormenta_id}): 0 - {vmax} (Step: {x})")

        fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': CART_PROJ})
        fig.suptitle(f'{nombre_modelo} — Validation storm {tormenta_id}', fontsize=15, y=0.98)

        ext_lon_min, ext_lon_max = float(np.min(LONS)), float(np.max(LONS))
        ext_lat_min, ext_lat_max = float(np.min(LATS)), float(np.max(LATS))

        def dibujar_fondo(ax):
            ax.set_facecolor('white')
            ax.set_extent([ext_lon_min, ext_lon_max, ext_lat_min, ext_lat_max], crs=crs.PlateCarree())
            ax.add_feature(cfeature.BORDERS, linewidth=0.8)
            ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
            ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xlabel_style = {'size': 8, 'color': 'gray'}
            gl.ylabel_style = {'size': 8, 'color': 'gray'}

        for ax in axes:
            dibujar_fondo(ax)

        pbar = tqdm(total=n_frames, desc="Rendering Animation")

        def update(frame_idx):
            for ax in axes:
                ax.cla()
                dibujar_fondo(ax)

            cf0 = axes[0].pcolormesh(LONS, LATS, reales_cache[frame_idx], transform=crs.PlateCarree(),
                                      cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax, shading='nearest')
            axes[0].set_title(f'Ground Truth: LIGHTDENS — Frame {frame_idx}')

            cf1 = axes[1].pcolormesh(LONS, LATS, predicciones_cache[frame_idx], transform=crs.PlateCarree(),
                                      cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax, shading='nearest')
            axes[1].set_title(f'Prediction: {nombre_modelo}')

            pbar.update(1)
            return cf0, cf1

        cf_inicial_real, cf_inicial_pred = update(0)

        fig.colorbar(cf_inicial_real, ax=axes[0], label='LIGHTDENS (flashes/column)', fraction=0.046, pad=0.04, ticks=ticks)
        fig.colorbar(cf_inicial_pred, ax=axes[1], label='LIGHTDENS (flashes/column)', fraction=0.046, pad=0.04, ticks=ticks)

        plt.tight_layout()

        ani = animation.FuncAnimation(fig, update, frames=range(n_frames), interval=200)

        carpeta_destino = "../videos&gifs"
        os.makedirs(carpeta_destino, exist_ok=True)
        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        nombre_archivo_salida = f"reconstruction_{nombre_limpio}_{tormenta_id}.gif"
        ruta_final = os.path.join(carpeta_destino, nombre_archivo_salida)

        ani.save(ruta_final, writer="pillow", fps=fps)
        pbar.close()
        plt.close(fig)

        print(f"Animation saved to {ruta_final}")
        rutas_generadas.append(ruta_final)

    return rutas_generadas
    
def mostrar_historial_tabular(checkpoint):
    """
    Toma el checkpoint de un modelo y muestra una tabla interactiva con 
    el historial de pérdidas (Train y Val) época por época.
    Resalta en verde la época con el menor error de validación.
    """
    # 1. Extraemos las listas del checkpoint
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    
    # 2. Armamos un DataFrame de Pandas
    df = pd.DataFrame({
        'Época': epocas,
        'Train Loss (HFACLV2.1)': historial_train,
        'Val Loss (HFACLV2.1)': historial_val
    })
    
    # 3. Calculamos la diferencia con la época anterior para ver si mejoró
    mejor_hasta_ayer = df['Val Loss (HFACLV2.1)'].shift(1).cummin().fillna(float('inf'))
    df['¿Mejoró?'] = (df['Val Loss (HFACLV2.1)'] < mejor_hasta_ayer).apply(lambda x: '🟢 Sí' if x else '🔴 No')
    df.loc[0, '¿Mejoró?'] = '🟢 Sí'
    
    # 4. Buscamos cuál fue la mejor época absoluta
    mejor_val_loss = min(historial_val)
    
    # 5. Función para pintar la fila ganadora de verde
    def resaltar_mejor_epoca(fila):
        if fila['Val Loss (HFACLV2.1)'] == mejor_val_loss:
            return ['background-color: #d4edda; color: #155724; font-weight: bold'] * len(fila)
        else:
            return [''] * len(fila)
            
    # 6. Aplicamos el formato (dejamos 6 decimales) y mostramos la tabla
    tabla_estilizada = df.style.apply(resaltar_mejor_epoca, axis=1) \
                               .format({'Train Loss (HFACLV2.1)': "{:.6f}", 
                                        'Val Loss (HFACLV2.1)': "{:.6f}"}) \
                               .hide(axis="index")
    
    print("HISTORIAL DE ENTRENAMIENTO TABULAR")
    print("-" * 40)
    display(tabla_estilizada)

    return df

def calcular_data_range_global(dataset):
    """
    Calcula el rango (max - min) de LIGHTDENS real sobre TODO el dataset.
    """
    max_global = float("-inf")
    min_global = float("inf")
    for idx in tqdm(range(len(dataset)), desc="Calculando data_range global"):
        _, y = dataset[idx]
        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        max_global = max(max_global, y_real.max())
        min_global = min(min_global, y_real.min())
    return max_global - min_global

def calcular_metricas_timestep(y_real, y_pred, data_range=None):
    """
    Calcula un set de métricas de verificación comparando la predicción
    contra LIGHTDENS real, para un único timestep. Todas las métricas
    se evalúan solo sobre los píxeles con actividad eléctrica (observada
    o predicha).
    """
    mascara = (y_real > 0) | (y_pred > 0)
    actividad_total = y_real.sum()  # se mide sobre el dominio completo, no filtrado

    if data_range is None:
        data_range = max(y_real.max(), y_pred.max()) - min(y_real.min(), y_pred.min())

    # Mapa de SSIM local, calculado sobre el campo completo
    if data_range > 0:
        _, mapa_ssim = ssim(y_real, y_pred, data_range=data_range, full=True)
    else:
        mapa_ssim = np.ones_like(y_real)

    # Si no hay actividad observada NI predicha, devolvemos NaN para que pandas lo ignore al graficar
    if mascara.sum() == 0:
        return {'mse': np.nan, 'rmse': np.nan, 'mae': np.nan, 'ssim': np.nan,
                'pod': np.nan, 'far': np.nan, 'csi': np.nan,
                'actividad': actividad_total}

    # Filtramos por la máscara
    y_real_m = y_real[mascara]
    y_pred_m = y_pred[mascara]

    # Métricas de error continuo
    error = y_real_m - y_pred_m
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(error))
    valor_ssim = mapa_ssim[mascara].mean()

    # Cálculo de las 3 Titánides Categóricas (POD, FAR, CSI)
    obs_bin = (y_real_m > 0)
    pred_bin = (y_pred_m > 0)

    hits = (obs_bin & pred_bin).sum()
    misses = (obs_bin & ~pred_bin).sum()
    false_alarms = (~obs_bin & pred_bin).sum()

    pod = hits / (hits + misses) if (hits + misses) > 0 else np.nan
    far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else np.nan
    csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else np.nan

    return {
        'mse': mse, 'rmse': rmse, 'mae': mae, 'ssim': valor_ssim,
        'pod': pod, 'far': far, 'csi': csi,
        'actividad': actividad_total
    }

def evaluar_dataset(modelo, dataset, data_range=None):
    modelo.eval()
    registros = []

    if data_range is None:
        data_range = calcular_data_range_global(dataset)
        print(f"Data range global (LIGHTDENS): {data_range:.2f}")

    grupos_por_tormenta = agrupar_indices_por_tormenta(dataset)
    tormenta_de_idx = {idx: tid for tid, idxs in grupos_por_tormenta.items() for idx in idxs}

    for idx in tqdm(range(len(dataset)), desc="Calculando métricas"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

        metricas = calcular_metricas_timestep(y_real, y_pred, data_range=data_range)
        metricas['idx'] = idx
        metricas['tormenta'] = tormenta_de_idx[idx]

        registros.append(metricas)

    return pd.DataFrame(registros)

def graficar_metrica(df_metricas, columna, nombre_modelo='Model', ylabel=None, mostrar_actividad=False):
    """
    Plots the evolution of a verification metric across the validation
    timesteps, coloring each validation storm differently and mapping
    storm ids in the legend (English labels).
    """
    # Determinamos qué columnas necesitamos extraer
    columnas_req = ['idx', columna, 'tormenta']
    if mostrar_actividad and 'actividad' in df_metricas.columns:
        columnas_req.append('actividad')
        
    serie = df_metricas[columnas_req].dropna(subset=[columna])

    with plt.style.context('seaborn-v0_8-darkgrid'):
        fig, ax = plt.subplots(figsize=(11, 5), facecolor='white')

        tormentas_unicas = serie['tormenta'].unique()
        colores = plt.cm.tab10(np.linspace(0, 1, len(tormentas_unicas)))

        # NUEVO: Crear eje secundario para la actividad eléctrica (fondo)
        ax2 = None
        if mostrar_actividad and 'actividad' in df_metricas.columns:
            ax2 = ax.twinx()
            ax2.bar(serie['idx'], serie['actividad'], width=1.0, color='gray', alpha=0.25, 
                    label='Electrical Activity (Total Flashes)')
            ax2.set_ylabel('Total Flashes (Ground Truth)', color='gray', fontweight='bold')
            ax2.tick_params(axis='y', labelcolor='gray')
            ax2.grid(False) # Apagamos la grilla secundaria para no ensuciar la principal

        # Graficamos la métrica principal
        for i, tormenta in enumerate(tormentas_unicas):
            grupo = serie[serie['tormenta'] == tormenta]
            ax.plot(
                grupo['idx'],
                grupo[columna],
                'o-',
                color=colores[i],
                markersize=4,
                linewidth=1.2,
                label=f"Storm {tormenta}",
                zorder=3 # Asegura que la línea quede por encima de las barras
            )

        media = serie[columna].mean()
        ax.axhline(
            media,
            color='black',
            linestyle='--',
            alpha=0.8,
            linewidth=1.5,
            label=f'Overall Mean: {media:.3f}',
            zorder=3
        )

        ax.set_xlabel('Validation Timestep')
        ax.set_ylabel(ylabel or columna.upper(), fontweight='bold')
        ax.set_title(f'{ylabel or columna.upper()} — {nombre_modelo}')

        # Combinar leyendas de ambos ejes si ax2 existe
        # (El bbox_to_anchor se corre a 1.15 para no superponerse con los números del eje secundario)
        if ax2:
            lines_1, labels_1 = ax.get_legend_handles_labels()
            lines_2, labels_2 = ax2.get_legend_handles_labels()
            ax.legend(lines_1 + lines_2, labels_1 + labels_2, 
                      bbox_to_anchor=(1.15, 1), loc='upper left', frameon=True, facecolor='white')
        else:
            ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white')
            
        plt.tight_layout()

        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        ruta_guardado = generar_ruta_imagen('../images', f"{columna}_{nombre_limpio}")

        plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Plot for {columna.upper()} saved to: {ruta_guardado}")

        plt.show()

    return media

def graficar_mse(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'mse', nombre_modelo, ylabel='MSE')

def graficar_rmse(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'rmse', nombre_modelo, ylabel='RMSE')

def graficar_mae(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'mae', nombre_modelo, ylabel='MAE')

def graficar_ssim(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'ssim', nombre_modelo, ylabel='SSIM', mostrar_actividad=True)

def graficar_pod(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'pod', nombre_modelo, ylabel='Probability of Detection (POD)', mostrar_actividad=True)

def graficar_far(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'far', nombre_modelo, ylabel='False Alarm Ratio (FAR)', mostrar_actividad=True)

def graficar_csi(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'csi', nombre_modelo, ylabel='Critical Success Index (CSI)', mostrar_actividad=True)

def calcular_r2_global(modelo, dataset, nombre_modelo='Model', ignorar_borde=0):
    """
    Calculates R² as a global metric over all active pixels
    in the dataset, for a single model.
    """
    modelo.eval()
    y_real_total = []
    y_pred_total = []

    for idx in tqdm(range(len(dataset)), desc=f"Calculating R² — {nombre_modelo}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

        if ignorar_borde > 0:
            y_real = y_real[ignorar_borde:-ignorar_borde, ignorar_borde:-ignorar_borde]
            y_pred = y_pred[ignorar_borde:-ignorar_borde, ignorar_borde:-ignorar_borde]

        mascara = (y_real > 0) | (y_pred > 0)
        if mascara.sum() > 0:
            y_real_total.append(y_real[mascara])
            y_pred_total.append(y_pred[mascara])

    y_real_all = np.concatenate(y_real_total)
    y_pred_all = np.concatenate(y_pred_total)

    ss_res = np.sum((y_real_all - y_pred_all) ** 2)
    ss_tot = np.sum((y_real_all - y_real_all.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    edge_label = f" (ignoring {ignorar_borde}px edge)" if ignorar_borde > 0 else " (full image)"
    print(f"R² {nombre_modelo}{edge_label} — over {len(y_real_all):,} active pixels: {r2:.4f}")

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    sufijo_borde = f"_noedge{ignorar_borde}" if ignorar_borde > 0 else "_full"
    ruta_log = f"../modelos/r2_{nombre_limpio}{sufijo_borde}.txt"
    
    with open(ruta_log, 'w', encoding='utf-8') as f:
        f.write(f"Model: {nombre_modelo}\n")
        f.write(f"Edge crop: {ignorar_borde}px\n")
        f.write(f"Active pixels evaluated: {len(y_real_all):,}\n")
        f.write(f"R²: {r2:.6f}\n")
    print(f"Result saved to: {ruta_log}")

    residuos = y_pred_all - y_real_all

    with plt.style.context('seaborn-v0_8-darkgrid'):
        # Añadido facecolor='white' a la figura también para asegurar consistencia
        fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
        ax.hist(residuos, bins=100, color='#722F37', edgecolor='none', alpha=0.85)
        ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7, label='Error = 0')
        ax.axvline(residuos.mean(), color='gold', linestyle='-', linewidth=2,
                   label=f'Mean bias: {residuos.mean():+.2f}')
        
        ax.set_xlim(-100, 100)
        
        ax.set_xlabel('Residuals (Predicted − Actual)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'Residuals Distribution — {nombre_modelo}{edge_label}')
        ax.legend()
        plt.tight_layout()

        ruta_hist = f"../images/residuals_{nombre_limpio}{sufijo_borde}.png"
        # Facecolor white al exportar
        plt.savefig(ruta_hist, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Histogram saved to: {ruta_hist}")
        plt.show()

    return r2

def graficar_picos_de_error(df_metricas, modelo, dataset, nombre_modelo='Modelo', top_n=3, metrica='mse', canal_entrada=0):
    """
    Encuentra los timesteps (índices) donde el modelo tuvo el mayor error 
    (los picos más altos) y grafica la reconstrucción para entender qué pasó.
    
    Parámetros
    ----------
    df_metricas   : DataFrame devuelto por evaluar_dataset().
    modelo        : modelo PyTorch ya cargado y en eval mode.
    dataset       : TormentaDataset (val_dataset).
    nombre_modelo : str, para el título.
    top_n         : int, cantidad de picos a mostrar (por defecto los 3 peores).
    metrica       : str, la columna del df a usar para buscar los picos ('mse', 'mae', etc).
    canal_entrada : int, índice del canal a mostrar en la gráfica de entrada.
        Percentil de los valores activos (LIGHTDENS > 0) utilizado como
        límite superior de la escala de color en las reconstrucciones.
        Valores menores achican más la escala y resaltan mejor la estructura
        espacial a costa de saturar los picos más intensos.
    """
    # Verificamos que la métrica exista en el DataFrame
    if metrica not in df_metricas.columns:
        raise ValueError(f"La métrica '{metrica}' no se encuentra en el DataFrame.")

    # Calculamos la media para dar un poco de contexto de "qué tan grave" es el pico
    media_metrica = df_metricas[metrica].mean()
    
    # Extraemos las 'top_n' filas con los valores más altos de la métrica
    df_picos = df_metricas.nlargest(top_n, metrica)
    
    print(f"{"="*60}")
    print(f"🔍 ANALIZANDO LOS {top_n} PEORES CASOS SEGÚN {metrica.upper()}")
    print(f"Media global de {metrica.upper()} en este dataset: {media_metrica:.4f}")
    print(f"{"="*60}\n")
    
    # Iteramos sobre esas peores filas
    for ranking, (_, fila) in enumerate(df_picos.iterrows(), start=1):
        idx_pico = int(fila['idx'])
        valor_pico = fila[metrica]
        
        # Calculamos cuántas veces peor es este pico en comparación a la media
        multiplo = valor_pico / media_metrica if media_metrica > 0 else 0
        
        print(f"⚠️ PICO #{ranking} | Timestep: {idx_pico}")
        print(f"-> {metrica.upper()}: {valor_pico:.4f} (El error es {multiplo:.1f} veces la media global)")
        
        # Graficamos el índice extraído
        graficar_reconstruccion(
            modelo=modelo,
            dataset=dataset,
            nombre_modelo=nombre_modelo,
            canal_entrada=canal_entrada,
            idx=idx_pico,
        )
        print("-" * 60)
        print("\n")

def resumen_pesos_loss(criterion):
    """
    Muestra los pesos aprendidos de Uncertainty Weighting al final
    del entrenamiento, convertidos a sigma^2 para interpretación directa.

    Valores más chicos de sigma^2 indican que ese término tuvo mayor
    peso relativo (más "confianza") durante el entrenamiento.
    """
    log_vars = {
        'WMSE': criterion.log_var_wmse.item(),
        'FAL':  criterion.log_var_fal.item(),
        'FCL':  criterion.log_var_fcl.item(),
    }

    print("PESOS APRENDIDOS (Uncertainty Weighting)")
    print("-" * 45)
    for nombre, log_var in log_vars.items():
        sigma2 = np.exp(log_var)
        precision = np.exp(-log_var)
        print(f"  {nombre:5s} — log_var: {log_var:+.4f} | σ²: {sigma2:.4f} | peso (precisión): {precision:.4f}")

    return log_vars

def graficar_distribucion_curvas(modelo, dataset, nombre_modelo='Modelo', bin_w=1.0, sigma=2.0):
    """Curvas de densidad (eje Y log) de Observed vs Predicted, sobre TODOS los píxeles."""
    modelo.eval()
    xmax = float(dataset.desnormalizar_y(torch.tensor(1.0)))   # máximo posible de la escala
    bins = np.arange(0, xmax + bin_w, bin_w)
    centros = 0.5 * (bins[:-1] + bins[1:])
    cnt_real = np.zeros(len(centros), dtype=np.float64)
    cnt_pred = np.zeros(len(centros), dtype=np.float64)
    pn_min, pn_max, pn_sum, n = np.inf, -np.inf, 0.0, 0   # stats de la salida normalizada

    for idx in tqdm(range(len(dataset)), desc=f"Recolectando — {nombre_modelo}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_norm = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy().ravel())
        y_pred = np.round(dataset.desnormalizar_y(pred_norm).numpy().ravel())
        cnt_real += np.histogram(y_real, bins=bins)[0]
        cnt_pred += np.histogram(y_pred, bins=bins)[0]

        pn = pred_norm.numpy().ravel()
        pn_min, pn_max = min(pn_min, pn.min()), max(pn_max, pn.max())
        pn_sum += pn.sum(); n += pn.size

    fig, ax = plt.subplots(figsize=(7, 5))
    for cnt, lbl, c in [(cnt_real, "Observed", "tab:blue"),
                        (cnt_pred, "Predicted", "tab:orange")]:
        if cnt.sum() == 0:
            continue
            
        dens = cnt / (cnt.sum() * bin_w)
        
        # --- APLICAR SUAVIZADO AQUÍ ---
        if sigma > 0:
            dens = gaussian_filter1d(dens, sigma=sigma)
            
        m = dens > 0
        ax.plot(centros[m], dens[m], label=lbl, color=c, lw=1.2)

    ax.set_yscale("log")
    ax.set_xlabel("flashes / columna")
    ax.set_ylabel("Probability Density")
    ax.set_title(f"Lightning distribution — {nombre_modelo}")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    plt.savefig(f"../images/dist_curvas_{nombre_limpio}.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Salida normalizada del modelo — min: {pn_min:.4f}, max: {pn_max:.4f}, media: {pn_sum/n:.4f}")
    return centros, cnt_real, cnt_pred


def graficar_distribucion_curvas_por_tormenta(modelo, dataset, nombre_modelo='Modelo', bin_w=1.0, sigma=2.0):
    """
    Curvas de densidad (eje Y log) de Observed vs Predicted, una por cada
    tormenta de `dataset` (detectadas automáticamente por salto temporal).
    """
    modelo.eval()
    xmax = float(dataset.desnormalizar_y(torch.tensor(1.0)))
    bins = np.arange(0, xmax + bin_w, bin_w)
    centros = 0.5 * (bins[:-1] + bins[1:])

    grupos = agrupar_indices_por_tormenta(dataset)
    resultados = {}

    for tormenta_id, indices in grupos.items():
        cnt_real = np.zeros(len(centros), dtype=np.float64)
        cnt_pred = np.zeros(len(centros), dtype=np.float64)
        pn_min, pn_max, pn_sum, n = np.inf, -np.inf, 0.0, 0

        for idx in tqdm(indices, desc=f"Recolectando — {tormenta_id}"):
            X, y = dataset[idx]
            with torch.no_grad():
                pred_norm = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

            y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy().ravel())
            y_pred = np.round(dataset.desnormalizar_y(pred_norm).numpy().ravel())
            cnt_real += np.histogram(y_real, bins=bins)[0]
            cnt_pred += np.histogram(y_pred, bins=bins)[0]

            pn = pred_norm.numpy().ravel()
            pn_min, pn_max = min(pn_min, pn.min()), max(pn_max, pn.max())
            pn_sum += pn.sum(); n += pn.size

        fig, ax = plt.subplots(figsize=(7, 5))
        for cnt, lbl, c in [(cnt_real, "Observed", "tab:blue"),
                            (cnt_pred, "Predicted", "tab:orange")]:
            if cnt.sum() == 0:
                continue
                
            dens = cnt / (cnt.sum() * bin_w)
            
            # --- APLICAR SUAVIZADO AQUÍ ---
            if sigma > 0:
                dens = gaussian_filter1d(dens, sigma=sigma)
                
            m = dens > 0
            ax.plot(centros[m], dens[m], label=lbl, color=c, lw=1.2)

        ax.set_yscale("log")
        ax.set_xlabel("flashes / columna")
        ax.set_ylabel("Probability Density")
        ax.set_title(f"Lightning distribution — {nombre_modelo} — Storm {tormenta_id}")
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()

        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        ruta = f"../images/dist_curvas_{nombre_limpio}_{tormenta_id}.png"
        plt.savefig(ruta, dpi=300, bbox_inches='tight')
        plt.show()

        print(f"Storm {tormenta_id} — salida normalizada: min={pn_min:.4f}, max={pn_max:.4f}, media={pn_sum/n:.4f}")

        resultados[tormenta_id] = {
            "centros": centros, "cnt_real": cnt_real, "cnt_pred": cnt_pred,
            "pn_min": pn_min, "pn_max": pn_max, "pn_mean": pn_sum / n,
        }

    return resultados

def calcular_matriz_fss_dataset(modelo, dataloader, dataset,
                                umbrales=(1, 5, 15, 30),
                                ventanas=(1, 3, 5, 15)):
    """
    FSS global en GPU, evaluado solo en los puntos con actividad eléctrica
    observada (LIGHTDENS > 0) ya sea por el modelo o por WRF-Elec. Acumula 
    sum((obs_frac - pred_frac)^2) y sum(obs_frac^2 + pred_frac^2) sobre 
    esos puntos y divide recién al final. Devuelve un np.ndarray de forma
    (len(ventanas), len(umbrales)).
    """
    assert all(v % 2 == 1 for v in ventanas), "Las ventanas deben ser impares (padding simétrico)."

    modelo.eval()
    U, V = len(umbrales), len(ventanas)
    umb = torch.tensor(umbrales, device=device, dtype=torch.float32).view(1, U, 1, 1)

    num = torch.zeros(V, U, device=device, dtype=torch.float64)
    den = torch.zeros(V, U, device=device, dtype=torch.float64)

    with torch.no_grad():
        for X, y in tqdm(dataloader, desc="FSS"):
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            p = modelo(X)

            y_real = torch.round(dataset.desnormalizar_y(y))
            y_pred = torch.round(dataset.desnormalizar_y(p))

            # Máscara de puntos con actividad eléctrica observada: (B, 1, H, W)
            mask = ((y_real > 0) | (y_pred > 0)).float()

            obs_bin  = (y_real >= umb).float()
            pred_bin = (y_pred >= umb).float()

            for i, v in enumerate(ventanas):
                # Las fracciones se calculan sobre el campo completo (con los vecinos)
                obs_frac  = F.avg_pool2d(obs_bin,  kernel_size=v, stride=1, padding=v // 2)
                pred_frac = F.avg_pool2d(pred_bin, kernel_size=v, stride=1, padding=v // 2)

                # ...pero el error se suma solo donde hay actividad (mask se broadcastea sobre U)
                num[i] += (((obs_frac - pred_frac) ** 2) * mask).sum(dim=(0, 2, 3)).double()
                den[i] += ((obs_frac ** 2 + pred_frac ** 2) * mask).sum(dim=(0, 2, 3)).double()

    fss = torch.where(den == 0, torch.ones_like(den), 1.0 - num / den)
    return fss.cpu().numpy()


def graficar_heatmap_fss(matriz, umbrales=(1, 5, 15, 30), ventanas=(1, 3, 5, 15),
                         nombre_modelo='Modelo'):
    plt.figure(figsize=(7, 5), facecolor='white')
    cmap = sns.diverging_palette(10, 130, as_cmap=True)
    ax = sns.heatmap(
        matriz, annot=True, fmt=".2f", cmap=cmap,
        vmin=0.0, vmax=1.0, center=0.5,
        xticklabels=umbrales, yticklabels=ventanas,
        linewidths=1, linecolor='white',
        cbar_kws={'label': 'Fractional Skill Score (FSS)'}
    )
    ax.invert_yaxis()
    plt.xlabel('Umbral de Intensidad (flashes/columna)', fontweight='bold')
    plt.ylabel('Escala Espacial / Ventana (px)', fontweight='bold')
    plt.title(f'Matriz FSS: Tolerancia Espacial vs Severidad — {nombre_modelo}', pad=15)
    plt.tight_layout()
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    plt.savefig(f"../images/fss_heatmap_{nombre_limpio}.png", dpi=300, bbox_inches='tight')
    plt.show()

def evaluar_ssim_ventanas(modelo, dataset, win_sizes=[3, 5, 7, 11, 15], data_range=None):
    """
    Evalúa el dataset con el modelo actual calculando SSIM para distintas
    ventanas espaciales (win_size). El SSIM se calcula sobre el campo
    completo (para no romper la ventana local) y se promedia solo sobre
    los píxeles con actividad eléctrica (observada o predicha).
    """
    modelo.eval()
    registros = []

    if data_range is None:
        data_range = calcular_data_range_global(dataset)
        print(f"Data range global (LIGHTDENS): {data_range:.2f}")

    grupos_por_tormenta = agrupar_indices_por_tormenta(dataset)
    tormenta_de_idx = {idx: tid for tid, idxs in grupos_por_tormenta.items() for idx in idxs}

    for idx in tqdm(range(len(dataset)), desc="Calculando SSIM multi-ventana"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

        mascara = (y_real > 0) | (y_pred > 0)

        registro = {
            'idx': idx,
            'tormenta': tormenta_de_idx[idx],
            'actividad': y_real.sum()
        }

        for w in win_sizes:
            if data_range > 0 and mascara.sum() > 0:
                _, mapa_ssim = ssim(y_real, y_pred, data_range=data_range, win_size=w, full=True)
                registro[f'ssim_{w}x{w}'] = mapa_ssim[mascara].mean()
            elif data_range <= 0:
                registro[f'ssim_{w}x{w}'] = 1.0
            else:
                # Sin actividad ni observada ni predicha: nada que promediar
                registro[f'ssim_{w}x{w}'] = np.nan

        registros.append(registro)

    return pd.DataFrame(registros)

def graficar_sensibilidad_ssim_ventanas(df_resultados, win_sizes=[3, 5, 7, 11, 15], nombre_modelo='Modelo'):
    # Transformamos el DataFrame a formato largo (melt) para seaborn
    columnas_ssim = [f'ssim_{w}x{w}' for w in win_sizes]
    df_long = df_resultados.melt(
        id_vars=['idx', 'tormenta'], 
        value_vars=columnas_ssim,
        var_name='Ventana', 
        value_name='SSIM'
    )

    with plt.style.context('seaborn-v0_8-darkgrid'):
        fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
        
        sns.boxplot(
            data=df_long, 
            x='Ventana', 
            y='SSIM', 
            ax=ax, 
            palette='Blues',
            showmeans=True,
            meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"red"}
        )

        # Imprimir las medias exactas en pantalla
        print("--- Medias de SSIM por ventana ---")
        for col in columnas_ssim:
            print(f"{col}: {df_resultados[col].mean():.4f}")

        ax.set_title(f"Análisis de Sensibilidad de SSIM según el Tamaño de Ventana — {nombre_modelo}", fontweight='bold', fontsize=13)
        ax.set_ylabel("SSIM", fontweight='bold')
        ax.set_xlabel("Tamaño de Ventana (win_size)", fontweight='bold')
        
        plt.tight_layout()
        
        # Limpiamos el nombre del modelo para usarlo en el nombre del archivo
        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        ruta_guardado = f'../images/sensibilidad_ssim_ventanas_{nombre_limpio}.png'
        
        plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
        print(f"Plot saved to: {ruta_guardado}")
        
        plt.show()

In [ ]:
resumen_entrenamiento(checkpoint, NOMBRE_MODELO, train_loader, val_loader, BATCH_SIZE)

In [ ]:
graficar_curva_error(checkpoint, nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_reconstruccion(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO, idx=None)

In [ ]:
rutas = animar_reconstruccion_por_tormenta(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO)

In [ ]:
df_historial = mostrar_historial_tabular(checkpoint)
ruta_txt = f"../modelos/historial_{NOMBRE_MODELO}.txt"

with open(ruta_txt, 'w', encoding='utf-8') as f:
    f.write("=== HISTORIAL DE ENTRENAMIENTO ===\n")
    f.write(df_historial.to_string(index=False))
    
print(f"¡Guardado en {ruta_txt}!")

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_metricas = evaluar_dataset(modelo, val_dataset, data_range=data_range_val)

In [ ]:
graficar_mse(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_rmse(df_metricas, nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_mae(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_pod(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_far(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_csi(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
# Ventana por default de 7x7
graficar_ssim(df_metricas, nombre_modelo=NOMBRE_MODELO)

In [ ]:
# Ventanas a probar (siempre impares)
ventanas_a_probar = [3, 5, 7, 11, 15]

df_ssim_ventanas = evaluar_ssim_ventanas(
    modelo=modelo, 
    dataset=val_dataset, 
    win_sizes=ventanas_a_probar
)

graficar_sensibilidad_ssim_ventanas(df_ssim_ventanas, ventanas_a_probar, NOMBRE_MODELO)

In [ ]:
r2_completo   = calcular_r2_global(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO, ignorar_borde=0)
r2_sin_bordes = calcular_r2_global(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO, ignorar_borde=10)

print(f"\nDiferencia: {r2_sin_bordes - r2_completo:+.4f}")

In [ ]:
matriz_fss = calcular_matriz_fss_dataset(modelo, val_loader, val_dataset)
graficar_heatmap_fss(matriz_fss, nombre_modelo=NOMBRE_MODELO)

In [ ]:
centros, cnt_real, cnt_pred = graficar_distribucion_curvas(modelo, val_dataset, NOMBRE_MODELO)

In [ ]:
resultados = graficar_distribucion_curvas_por_tormenta(modelo, val_dataset, NOMBRE_MODELO)

In [ ]:
graficar_picos_de_error(df_metricas, modelo, val_dataset, nombre_modelo=NOMBRE_MODELO)

In [ ]:
# Todas las métricas por muestra, para subirlas al repo (results/ext_clip_u-net_comb_v2.1/)
os.makedirs('../results/ext_clip_u-net_comb_v2.1', exist_ok=True)
df_metricas.to_csv('../results/ext_clip_u-net_comb_v2.1/metricas_por_muestra.csv', index=False)
df_ssim_ventanas.to_csv('../results/ext_clip_u-net_comb_v2.1/ssim_ventanas.csv', index=False)
pd.DataFrame(matriz_fss).to_csv('../results/ext_clip_u-net_comb_v2.1/matriz_fss.csv')
print(df_metricas.describe().T)
print(f"R2 completo {r2_completo:.4f} | sin bordes {r2_sin_bordes:.4f}")


In [ ]:
# =====================================================================
# Reúne todos los resultados del modelo en ../resultados_ext_clip_u-net_comb_v2.1/ (y su .zip), con la
# misma estructura que los demás modelos de results/. Correr DESPUÉS de las celdas anteriores:
#   historial_entrenamiento.txt | curva_error.png | metricas/ (PNG, R2 en txt, CSV) | distribucion_curvas/
#   animaciones/ (GIF -> MP4 con ffmpeg, si está disponible) | reconstrucciones/
# Si una figura se generó varias veces (nombre, nombre_01...), se toma la última.
# =====================================================================
import re, glob, shutil, subprocess, zipfile   # os ya está importado

NOMBRE = "ext_clip_u-net_comb_v2.1"
NOMBRE_MODELO = "Ext Clip U-Net Comb V2.1"
SALIDA = f"../resultados_{NOMBRE}"
IMG, MOD, VID, EST, RES = "../images", "../modelos", "../videos&gifs", "../estudios", f"../results/{NOMBRE}"
SIN_MP4 = False        # True: dejar las animaciones en GIF (no convertir a MP4)
faltan = []


def ultima(carpeta, base, ext):
    """Archivo base.ext o la versión con contador más alto (base_01.ext, base_02.ext...)."""
    cands = []
    for ruta in glob.glob(os.path.join(carpeta, f"{glob.escape(base)}*{ext}")):
        m = re.fullmatch(re.escape(base) + r"(?:_(\d+))?" + re.escape(ext), os.path.basename(ruta))
        if m:
            cands.append((int(m.group(1) or 0), ruta))
    return max(cands)[1] if cands else None


def copiar(origen, destino, requerido=True):
    if origen is None or not os.path.exists(origen):
        if requerido:
            faltan.append(destino)
        return False
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    shutil.copy2(origen, destino)
    print(f"OK  {destino}")
    return True


if os.path.exists(SALIDA):
    shutil.rmtree(SALIDA)
os.makedirs(SALIDA)

# --- historial y curva de error ---
copiar(os.path.join(MOD, f"historial_{NOMBRE_MODELO}.txt"), f"{SALIDA}/historial_entrenamiento.txt")
copiar(ultima(IMG, f"curva_error_{NOMBRE}", ".png"), f"{SALIDA}/curva_error.png")

# --- métricas ---
for m in ("mse", "rmse", "mae", "ssim", "pod", "far", "csi"):
    copiar(ultima(IMG, f"{m}_{NOMBRE}", ".png"), f"{SALIDA}/metricas/{m}.png")
copiar(ultima(IMG, f"fss_heatmap_{NOMBRE}", ".png"), f"{SALIDA}/metricas/fss_heatmap.png")
copiar(ultima(IMG, f"sensibilidad_ssim_ventanas_{NOMBRE}", ".png"), f"{SALIDA}/metricas/sensibilidad_ssim_ventanas.png")
for suf in ("full", "noedge10"):
    copiar(ultima(IMG, f"residuals_{NOMBRE}_{suf}", ".png"), f"{SALIDA}/metricas/residuals_{suf}.png")
    copiar(os.path.join(MOD, f"r2_{NOMBRE}_{suf}.txt"), f"{SALIDA}/metricas/r2_{suf}.txt")
for csv in ("metricas_por_muestra.csv", "ssim_ventanas.csv", "matriz_fss.csv"):
    copiar(os.path.join(RES, csv), f"{SALIDA}/metricas/{csv}")

# --- distribución de curvas ---
copiar(ultima(IMG, f"dist_curvas_{NOMBRE}", ".png"), f"{SALIDA}/distribucion_curvas/dist_curvas.png")
tormentas = sorted({re.fullmatch(rf"dist_curvas_{re.escape(NOMBRE)}_(\d{{4}}-\d{{2}}-\d{{2}}_\d{{4}})(?:_\d+)?\.png",
                                 os.path.basename(r)).group(1)
                    for r in glob.glob(os.path.join(IMG, f"dist_curvas_{NOMBRE}_*.png"))
                    if re.fullmatch(rf"dist_curvas_{re.escape(NOMBRE)}_(\d{{4}}-\d{{2}}-\d{{2}}_\d{{4}})(?:_\d+)?\.png",
                                    os.path.basename(r))})
if not tormentas:
    faltan.append("distribucion_curvas/dist_curvas_<tormenta>.png")
for t in tormentas:
    copiar(ultima(IMG, f"dist_curvas_{NOMBRE}_{t}", ".png"), f"{SALIDA}/distribucion_curvas/dist_curvas_{t}.png")

# --- reconstrucciones (PNG) ---
for ruta in sorted(glob.glob(os.path.join(IMG, f"reconstruccion_{NOMBRE}_idx_*.png"))):
    copiar(ruta, f"{SALIDA}/reconstrucciones/{os.path.basename(ruta).replace(f'reconstruccion_{NOMBRE}_', 'reconstruccion_')}",
           requerido=False)

# --- animaciones (GIF -> MP4, como en los demás modelos) ---
gifs = sorted(glob.glob(os.path.join(VID, f"reconstruction_{NOMBRE}_*.gif")))
if not gifs:
    faltan.append("animaciones/reconstruction_<tormenta>.mp4")
hay_ffmpeg = shutil.which("ffmpeg") is not None
for g in gifs:
    t = os.path.basename(g)[len(f"reconstruction_{NOMBRE}_"):-len(".gif")]
    os.makedirs(f"{SALIDA}/animaciones", exist_ok=True)
    mp4 = f"{SALIDA}/animaciones/reconstruction_{t}.mp4"
    if hay_ffmpeg and not SIN_MP4:
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", g, "-movflags", "faststart",
                            "-pix_fmt", "yuv420p", "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
                            "-c:v", "libx264", "-crf", "23", mp4])
        if r.returncode == 0:
            print(f"OK  {mp4}")
            continue
        print(f"AVISO: ffmpeg falló con {g}; se deja el GIF")
    copiar(g, f"{SALIDA}/animaciones/reconstruction_{t}.gif")
if gifs and not hay_ffmpeg and not SIN_MP4:
    print("AVISO: no hay ffmpeg en este entorno; las animaciones quedaron en GIF "
          "(convertilas en tu compu: ffmpeg -i x.gif -movflags faststart -pix_fmt yuv420p "
          "-vf scale=trunc(iw/2)*2:trunc(ih/2)*2 x.mp4).")

# --- zip ---
ruta_zip = f"{SALIDA}.zip"
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for raiz, _, archivos in os.walk(SALIDA):
        for a in archivos:
            ruta = os.path.join(raiz, a)
            z.write(ruta, os.path.join(NOMBRE, os.path.relpath(ruta, SALIDA)))
print(f"\nZIP: {ruta_zip} ({os.path.getsize(ruta_zip)/1e6:.1f} MB)")
if faltan:
    print("\nFALTAN (corré en el notebook la celda que los genera y repetí):")
    for f in faltan:
        print("  -", f)
else:
    print("Está todo.")
